# Curadoria - Ajuste de máscaras dataset de segmentação Coral Sol

**Autor:** Breno Krohling

**Data:** 12-03-2026

## Objetivo
Realizar um ajuste nas máscaras de imagens do dataset de coral sol que estão extrapolando os limites das imagens

## Imports necessários

In [ ]:
from shapely.geometry import Polygon, box
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import clear_output
import os
from tqdm import tqdm
from PIL import Image


## Contantes

In [ ]:
IMAGES_DIR = 'curadoria_coral_sol\\imagens'
LABELS_DIR = 'curadoria_coral_sol\\labels'

## Funções

In [ ]:
def list_image_label_pairs(images_dir, labels_dir):
    """
    Lists pairs of image and label files with matching names in the given directories.
    Args:
        images_dir (str): Path to the directory containing images.
        labels_dir (str): Path to the directory containing label files.
    Returns:
        list of tuples: Each tuple contains (image_path, label_path).
    """
    pairs = []
    for image_name in os.listdir(images_dir):
        if image_name.lower().endswith(('.jpg', '.png', '.jpeg')):
            label_name = os.path.splitext(image_name)[0] + '.txt'
            image_path = os.path.join(images_dir, image_name)
            label_path = os.path.join(labels_dir, label_name)
            if os.path.exists(label_path):
                pairs.append((image_path, label_path))
    return pairs
    
def read_segmentation_txt_shapely(txt_path, image_shape):
    """
    Reads a segmentation label file with normalized coordinates and returns a list of polygons and their class IDs.
    Args:
        txt_path (str): Path to the label file.
        image_shape (tuple): (height, width) of the corresponding image.
    Returns:
        list of dicts: Each dict contains 'class_id' (int) and 'polygon' (shapely.geometry.Polygon).
    """
    height, width = image_shape
    masks = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3:
                continue  # Skip empty or malformed lines
            class_id = int(parts[0])
            coords = list(map(float, parts[1:]))
            points = []
            for i in range(0, len(coords), 2):
                x = coords[i] * width
                y = coords[i+1] * height
                points.append((x, y))
            if len(points) >= 3:
                polygon = Polygon(points)
                masks.append({'class_id': class_id, 'polygon': polygon})
    return masks
    
def create_dataset_df(pairs):
    """
    Creates a dataset of image, mask, id tuples in a dataframe.
    Args:
        pairs (list): List of (image_path, label_path) tuples.
    Returns:
        DataFrame: Dataframe with columns image_path, split, segments which is a list of dicts containing a mask and its class id
    """

    dataset_data = {
        'image_path': [],
        'masks': []
    }

    for img_path, label_path in pairs:
        img = cv2.imread(img_path)
        height, width = img.shape[:2]
        masks = read_segmentation_txt_shapely(label_path, (height, width))
        dataset_data['image_path'].append(img_path)
        dataset_data['masks'].append(masks)
    return pd.DataFrame(data=dataset_data)

def clip_polygon_to_image(caminho_imagem: str, poligono: Polygon, area: int = 100):
    """
    Recorta (clipa) um polígono para que ele fique restrito aos limites de uma imagem.
    
    Args:
        caminho_imagem (str): Caminho base da imagem de referência
        poligono (Polygon): Polígono a ser ajustado, representando a máscara original.
        area (int): área mínima de um polígono. Caso seja menor, será removido.
    
    Returns:
        List[Polygon]: Lista de polígonos resultantes da interseção entre o polígono original e os limites da imagem.
        Retorna uma lista vazia caso não haja interseção.
    
    """
    with Image.open(caminho_imagem) as img:
        largura, altura = img.size
    
    imagem_bbox = box(0, 0, largura, altura)
    intersecao = poligono.intersection(imagem_bbox)
    
    if intersecao.is_empty:
            return []
    if isinstance(intersecao, Polygon):
        return [intersecao] if intersecao.area >= area else []
    # Se for MultiPolygon ou GeometryCollection, retorna todos os polígonos
    return [geom for geom in getattr(intersecao, 'geoms', []) 
            if isinstance(geom, Polygon) and geom.area >= area]

def visualize_segmentation(image_path, segments):
    """
    Displays an image with its segmentation polygons overlaid.
    Args:
        image_path (str): Path to the image file.
        segments (list): List of dicts with 'class_id' and 'polygon' (shapely).
    """
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10,14))
    plt.imshow(img)
    for seg in segments:
        x, y = seg['polygon'].exterior.xy
        plt.plot(x, y, color='red')
    plt.axis('off')
    plt.show()     

def salvar_labels_para_dataset(dataset, pasta_saida):
    """
    Para cada linha do DataFrame, salva um arquivo .txt com os labels normalizados.
    O nome do arquivo de label será igual ao da imagem, trocando a extensão por .txt.
    
    Args:
    
        dataset (DataFrame): DataFrame com as colunas:
            - 'image_path': caminho da imagem
            - 'masks': lista de dicts {'class_id': int, 'polygon': Polygon}
        pasta_saida (str): Pasta onde os arquivos .txt serão salvos.
    """
    
    os.makedirs(pasta_saida, exist_ok=True)
    for _, row in dataset.iterrows():
        caminho_imagem = row['image_path']
        lista_poligonos = row['masks']
        
        with Image.open(caminho_imagem) as img:
            largura, altura = img.size
        
        nome_base = os.path.splitext(os.path.basename(caminho_imagem))[0]
        caminho_label = os.path.join(pasta_saida, nome_base + '.txt')
      
        with open(caminho_label, 'w') as f:
            for item in lista_poligonos:
                class_id = item['class_id']
                poly = item['polygon']
                coords = list(poly.exterior.coords)[:-1]
                linha = [str(class_id)]
                for x, y in coords:
                    x_norm = x / largura
                    y_norm = y / altura
                    linha.append(str(x_norm))
                    linha.append(str(y_norm))
                f.write(' '.join(linha) + '\n')

## Curadoria de Máscaras


In [ ]:
pares_dados = list_image_label_pairs(IMAGES_DIR, LABELS_DIR)
dataset = create_dataset_df(pares_dados)
dataset.head()

for idx_row, row in dataset.iterrows():
    novos_poligonos = []
    for item in row['masks']:
        novos = clip_polygon_to_image(row['image_path'], item['polygon'], area=200)
        for p in novos:
            novos_poligonos.append({
                'class_id': item.get('class_id', 0),
                'polygon': p
            })
    dataset.at[idx_row, 'masks'] = novos_poligonos
salvar_labels_para_dataset(dataset, 'labels/')